<a href="https://colab.research.google.com/github/Ambuj-N/ai-tools/blob/main/Sigmoid_Attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [40]:
import torch
import math
import torch.nn.functional as F

In [41]:
def simple_sigmoid_attention(query, key, value):
  d_k=query.size(-1)
  similarity=torch.matmul(query,key.transpose(-2,-1))
  normalized=similarity/math.sqrt(d_k)
  attention_weight=torch.sigmoid(normalized)
  result=torch.matmul(attention_weight,value)
  return attention_weight,result

In [42]:
torch.manual_seed(1)

In [43]:
batch_size=1
seq_length=4
d_k=8

In [44]:
query=torch.randn(batch_size,seq_length,d_k)
key=torch.randn(batch_size,seq_length,d_k)
value=torch.randn(batch_size,seq_length,d_k)

In [45]:
attention_weight,result=simple_sigmoid_attention(query,key,value)

In [46]:
def stable_sigmoid_attention(query,key,value):
  d_k=query.size(-1)
  n=query.size(-2)
  query_norm=F.layer_norm(query,[d_k])
  key_norm=F.layer_norm(query,[d_k])
  bias=-math.log(n)
  similarity=torch.matmul(query_norm,key_norm.transpose(-2,-1))
  normalized=similarity/math.sqrt(d_k)
  attention_weight=torch.sigmoid(normalized+bias)
  result=torch.matmul(attention_weight,value)
  return attention_weight,result

In [47]:
stable_attention_weight,stable_result=stable_sigmoid_attention(query,key,value)

In [48]:
print("Unstable_attention_weights:\n")
print(attention_weight)
print("\nStable_Attention_Weights:\n")
print(stable_attention_weight)

Unstable_attention_weights:

tensor([[[0.4037, 0.4970, 0.2529, 0.8580],
         [0.3751, 0.3372, 0.7198, 0.2978],
         [0.3646, 0.4873, 0.6374, 0.6113],
         [0.7271, 0.4853, 0.3259, 0.8416]]])

Stable_Attention_Weights:

tensor([[[0.8088, 0.2835, 0.2967, 0.1437],
         [0.2835, 0.8088, 0.1968, 0.2304],
         [0.2967, 0.1968, 0.8088, 0.0846],
         [0.1437, 0.2304, 0.0846, 0.8088]]])


In [49]:
# print("Query_Shape: ",query.shape)
# print("Key_transpose_Shape: ",key.transpose(-2,-1).shape)
# print("Attention_Weight_Shape: ",attention_weight.shape)
# print("Value_Shape: ",value.shape)
# print("Result_Shape: ",result.shape)

In [50]:
# print("Query_Shape: ",query.shape)
# print("Key_transpose_Shape: ",key.transpose(-2,-1).shape)
# print("Attention_Weight_Shape: ",attention_weight.shape)
# print("Value_Shape: ",value.shape)
# print("Result_Shape: ",result.shape)

In [82]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class SigmoidAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        # self.layerscale = nn.Parameter(1e-4 * torch.ones(d_model))
        self.layerscale=nn.Parameter(1e-1* torch.ones(d_model))
        # self.layerscale=1

    def forward(self, x):
        seq_length = x.size(1)

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q_norm = F.layer_norm(q, [self.d_model])
        k_norm = F.layer_norm(k, [self.d_model])

        similarity = torch.matmul(q_norm, k_norm.transpose(-2, -1))
        scaled_scores = similarity / math.sqrt(self.d_model)

        # bias = -math.log(seq_length)
        bias=0

        biased_scores = scaled_scores + bias

        attention_weight = torch.sigmoid(biased_scores)
        attn_output = torch.matmul(attention_weight, v)

        projected_output = self.out_proj(attn_output)

        return projected_output * self.layerscale

print("SigmoidAttention Layer is ready!")

SigmoidAttention Layer is ready!


In [55]:
class SigmoidTransformerBlock(nn.Module):
    def __init__(self, d_model):
        super().__init__()

        self.attention = SigmoidAttention(d_model)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Linear(d_model * 4, d_model)
        )

    def forward(self, x):
        attn_output = self.attention(self.norm1(x))
        x = x + attn_output

        ffn_output = self.ffn(self.norm2(x))
        x = x + ffn_output

        return x

print("Transformer Block is ready!")

Transformer Block is ready!


In [57]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class SoftmaxAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        similarity = torch.matmul(q, k.transpose(-2, -1))
        scaled_scores = similarity / math.sqrt(self.d_model)

        attention_weight = F.softmax(scaled_scores, dim=-1)

        attn_output = torch.matmul(attention_weight, v)
        return self.out_proj(attn_output)

class SoftmaxTransformerBlock(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.attention = SoftmaxAttention(d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Linear(d_model * 4, d_model)
        )

    def forward(self, x):
        attn_output = self.attention(self.norm1(x))
        x = x + attn_output
        ffn_output = self.ffn(self.norm2(x))
        x = x + ffn_output
        return x

print("Softmax Baseline is ready!")

Softmax Baseline is ready!


In [58]:
def get_batch(batch_size=32, seq_length=16, d_model=8):
    data = torch.randn(batch_size, seq_length, d_model)
    return data, data

print("Dataset generator is ready!")

Dataset generator is ready!


In [83]:
import torch
import torch.nn as nn

d_model = 8
learning_rate = 0.0005
iterations = 300

sigmoid_model = SigmoidTransformerBlock(d_model)
softmax_model = SoftmaxTransformerBlock(d_model)

criterion = nn.MSELoss()
opt_sigmoid = torch.optim.Adam(sigmoid_model.parameters(), lr=learning_rate)
opt_softmax = torch.optim.Adam(softmax_model.parameters(), lr=learning_rate)

for i in range(iterations):
    x, y = get_batch(batch_size=32, seq_length=16, d_model=d_model)

    opt_sigmoid.zero_grad()
    out_sig = sigmoid_model(x)
    loss_sig = criterion(out_sig, y)
    loss_sig.backward()
    opt_sigmoid.step()

    opt_softmax.zero_grad()
    out_soft = softmax_model(x)
    loss_soft = criterion(out_soft, y)
    loss_soft.backward()
    opt_softmax.step()

    if (i + 1) % 5 == 0:
        print(f"Step {i+1:<4} | Sigmoid Loss: {loss_sig.item():.4f} | Softmax Loss: {loss_soft.item():.4f}")

Step 5    | Sigmoid Loss: 0.0547 | Softmax Loss: 0.0779
Step 10   | Sigmoid Loss: 0.0427 | Softmax Loss: 0.0686
Step 15   | Sigmoid Loss: 0.0386 | Softmax Loss: 0.0603
Step 20   | Sigmoid Loss: 0.0349 | Softmax Loss: 0.0558
Step 25   | Sigmoid Loss: 0.0284 | Softmax Loss: 0.0464
Step 30   | Sigmoid Loss: 0.0255 | Softmax Loss: 0.0414
Step 35   | Sigmoid Loss: 0.0235 | Softmax Loss: 0.0373
Step 40   | Sigmoid Loss: 0.0201 | Softmax Loss: 0.0320
Step 45   | Sigmoid Loss: 0.0187 | Softmax Loss: 0.0299
Step 50   | Sigmoid Loss: 0.0152 | Softmax Loss: 0.0259
Step 55   | Sigmoid Loss: 0.0138 | Softmax Loss: 0.0239
Step 60   | Sigmoid Loss: 0.0126 | Softmax Loss: 0.0208
Step 65   | Sigmoid Loss: 0.0116 | Softmax Loss: 0.0183
Step 70   | Sigmoid Loss: 0.0104 | Softmax Loss: 0.0161
Step 75   | Sigmoid Loss: 0.0088 | Softmax Loss: 0.0144
Step 80   | Sigmoid Loss: 0.0086 | Softmax Loss: 0.0145
Step 85   | Sigmoid Loss: 0.0073 | Softmax Loss: 0.0138
Step 90   | Sigmoid Loss: 0.0068 | Softmax Loss: